In [13]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append('../')
from utils import * #only needed for xgboost
from friedman1 import *

import warnings
warnings.filterwarnings("ignore") #just to supress warnings

In [14]:
#Function that returns best params for each pair of (d, target_instance) 
# and their average val rmse, rmse, mae, given a set of config columns (depending on method)
# This funcion can take a dataframe or a path to a csv as input.
# the best settings are chosen using the lowest val rmse
def topk_per_d_per_method(data, config_cols, k=5):
    """
    Return up to the top-k configs per (target_instances, d), ranked by avg_rmse.
    Averages are computed across seeds.
    """
    df = pd.read_csv(data, index_col=[0]) if isinstance(data, str) else data.copy()
    df = df.sort_values(by=['seed'] + config_cols)

    # aggregate over seeds
    agg = (df.groupby(config_cols, as_index=False)
             .agg(avg_rmse=('rmse', 'mean'),
                  avg_mae=('val_mae', 'mean'),
                  n_seeds=('seed', 'nunique')))

    # merge back on config_cols only (minimal fix)
    agg = agg.merge(df, on=config_cols, how='left')

    # sort globally by the fields that determine the ranking
    agg = agg.sort_values(by=['d', 'avg_rmse'], ascending=True)

    # select top-k per (target_instances, d)
    topk_df = agg.groupby(['d'], group_keys=False).head(k)

    # final compact output
    topk_d = topk_df[['d', 'avg_rmse', 'rmse', 'mae']]

    return topk_d, topk_df




In [15]:
#Read data (remove v = 0.05 because we chose not to include it)
data_xgboost = pd.read_csv('results/xgb.csv')
data_xgboost_warmstart = pd.read_csv('results/ttb_LS.csv')

# Here we make final vizes!

In [16]:

plt.figure(figsize = (18,18))

import matplotlib as mpl


best_xgboost, best_xgboost_params = topk_per_d_per_method(
    data_xgboost,
    ['d', 'v', 'target_tree_size'], k=10000
)
best_xgboost['Method'] = 'XGBoost'

best_xgboost_warmstart, best_xgboost_warmstart_params = topk_per_d_per_method(
    data_xgboost_warmstart,
    config_cols, k=5
)
best_xgboost_warmstart['Method'] = 'XGBoost Warmstart'


# create new df for viz
df = pd.concat([best_xgboost, best_xgboost_warmstart])
df = best_xgboost
sns.lineplot(data=df, x='d', y='avg_rmse', hue='Method')

NameError: name 'config_cols' is not defined

<Figure size 1800x1800 with 0 Axes>

In [ ]:
best_xgboost_params[0:50]

,d,v,target_tree_size,avg_rmse,avg_mae,n_seeds,Unnamed: 0,seed,val_rmse,val_mae,rmse,mae
350,1.0,0.300,1.0,1.320082,1.044590,5,70,900.0,1.277461,1.021699,1.243878,0.995132
351,1.0,0.300,1.0,1.320082,1.044590,5,1610,901.0,1.282820,1.009998,1.308790,1.063810
352,1.0,0.300,1.0,1.320082,1.044590,5,3150,902.0,1.279553,0.997711,1.262069,0.996270
353,1.0,0.300,1.0,1.320082,1.044590,5,4690,903.0,1.326093,1.063285,1.399178,1.118244
354,1.0,0.300,1.0,1.320082,1.044590,5,6230,904.0,1.422919,1.130257,1.386496,1.100086
175,1.0,0.175,1.0,1.337719,1.064625,5,35,900.0,1.292783,1.038800,1.254724,1.007612
176,1.0,0.175,1.0,1.337719,1.064625,5,1575,901.0,1.280167,1.009088,1.306842,1.070414
177,1.0,0.175,1.0,1.337719,1.064625,5,3115,902.0,1.303701,1.020008,1.287932,1.023300
178,1.0,0.175,1.0,1.337719,1.064625,5,4655,903.0,1.385067,1.118782,1.445364,1.154250
179,1.0,0.175,1.0,1.337719,1.064625,5,6195,904.0,1.422581,1.136447,1.393734,1.108113
